# Batch inference for OCR line anchors

`Models/best.pt` is a YOLO11m instance-segmentation checkpoint with a single `line` class and `reg_max=32`. The runner exports the box head's predictions in decoded page pixels. It uses batched FP16 on CUDA, overlapping CPU decode and GPU inference. The starting settings are 1024 pixels and 0.38 confidence. [BDRC model card](https://huggingface.co/BDRC/TiBLINE-yolo11m-seg)

Run this notebook from the repository root with its Python environment. Set the folder of original page images or add a UTF-8 manifest with one path per line. The full-run cell requires a configured corpus. The benchmark and preview also work with the local sample images.

In [ ]:
from pathlib import Path

from batch_line_boxes import InferenceOptions, benchmark, export_line_boxes, read_page

SOURCE_DIR = None  # Example: Path(r"D:\Datasets\pages")
SOURCE_MANIFEST = None  # Paths can be absolute or relative to SOURCE_DIR.
OUTPUT_DIR = Path("runs/line_boxes")
PREVIEW_DIR = Path("runs/line_boxes_preview")
ESTIMATED_PAGES = 180_000

OPTIONS = InferenceOptions(
    checkpoint=Path("Models/best.pt"),
    img_size=1024,
    confidence=0.38,
    iou=0.5,
    device="0",
    decode_workers=8,
)
BATCH_SIZES = (4, 8, 16, 24, 32)
BATCH_SIZE = 16
PAGES_PER_SHARD = 512

local_sample = Path("data/tiblined_yolo/images/train")
sample_root = SOURCE_DIR or (
    local_sample if local_sample.is_dir() else Path("TestImages")
)
sample_manifest = SOURCE_MANIFEST if SOURCE_DIR is not None else None

## Tune the GPU batch size

The pilot processes up to 128 evenly spaced pages from the selected source. Its throughput includes image reads and decoding, model inference and box extraction. It chooses the fastest measured batch size. The duration estimate assumes comparable files and storage for the full run.

In [ ]:
stats = benchmark(
    sample_root,
    OPTIONS,
    batch_sizes=BATCH_SIZES,
    sample_size=128,
    manifest=sample_manifest,
)
best = max(stats, key=lambda item: item["pages_per_second"])
BATCH_SIZE = best["batch_size"]
hours = ESTIMATED_PAGES / best["pages_per_second"] / 3600
print(
    f"Batch size {BATCH_SIZE} · estimated {hours:.1f} hours for {ESTIMATED_PAGES:,} pages"
)

## Review a few pages first

Check that boxes cover full Tibetan lines and that marginal text is handled as expected. Predicted boxes are pseudo-labels. Choose a new preview output directory when changing thresholds, model, or source.

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.dataset as ds
from matplotlib.patches import Rectangle

preview_shards = export_line_boxes(
    sample_root,
    PREVIEW_DIR,
    OPTIONS,
    batch_size=BATCH_SIZE,
    manifest=sample_manifest,
    limit=8,
)
preview = ds.dataset(preview_shards, format="parquet").to_table().to_pylist()
preview_rows = [row for row in preview if row["status"] == "ok"][:4]
if not preview_rows:
    raise RuntimeError("None of the preview pages could be decoded.")

fig, axes = plt.subplots(
    len(preview_rows), 1, squeeze=False, figsize=(14, 3 * len(preview_rows))
)
for ax, row in zip(axes[:, 0], preview_rows, strict=True):
    image = read_page(sample_root / row["image_path"])
    ax.imshow(image[:, :, ::-1])
    for line in row["lines"]:
        x, y = line["x_min"], line["y_min"]
        width, height = line["x_max"] - x, line["y_max"] - y
        ax.add_patch(
            Rectangle((x, y), width, height, fill=False, edgecolor="lime", linewidth=1)
        )
    ax.set_title(f"{row['image_path']} · {len(row['lines'])} lines")
    ax.axis("off")
plt.tight_layout()

## Process the corpus

`run.json` freezes the source list and records the checkpoint hash and inference settings. Each completed Parquet shard is published atomically. Rerun the cell with the same output directory to resume. Batch size and decode worker count can change between attempts; use a new output directory for new model or threshold settings. For an overnight run, the equivalent entry point is `python batch_line_boxes.py <SOURCE_DIR> --output runs/line_boxes --batch-size <BATCH_SIZE>`.

In [ ]:
if SOURCE_DIR is None:
    raise ValueError(
        "Set SOURCE_DIR to the page image directory before starting the full run."
    )

page_shards = export_line_boxes(
    SOURCE_DIR,
    OUTPUT_DIR,
    OPTIONS,
    batch_size=BATCH_SIZE,
    pages_per_shard=PAGES_PER_SHARD,
    manifest=SOURCE_MANIFEST,
)
print(page_shards)

## Export line anchors for OCR

The Parquet output preserves the full relative page path, decoded image dimensions, a geometric line index, the clipped pixel box and YOLO confidence. Box maxima are exclusive, so `image[y_min:y_max, x_min:x_max]` extracts a line crop from the same decoded page. Index order follows vertical box centers and breaks ties left-to-right. Multi-column or marginal text may need separate reading-order alignment.

Use these anchors to crop lines or align them to existing transcriptions before creating LightOnOCR fine-tuning examples. Any coordinate targets should use the format of the fine-tuning setup you choose. For PaddleOCR recognition, pair line crops with their transcripts. For PaddleOCR detection, the four clockwise corners of each box can become a polygon after alignment to the transcript. [LightOnOCR-2 training overview](https://huggingface.co/blog/lightonai/lightonocr-2) · [PaddleOCR training data formats](https://www.paddleocr.ai/main/en/datasets/ocr_datasets.html)

In [ ]:
from collections import Counter

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

page_dataset = ds.dataset(page_shards, format="parquet")
flat_path = OUTPUT_DIR / "lines.parquet"
temporary_path = flat_path.with_suffix(".tmp")
page_columns = ["image_path", "width", "height"]
line_columns = ["line_index", "x_min", "y_min", "x_max", "y_max", "confidence"]
flat_schema = pa.schema(
    [(name, page_dataset.schema.field(name).type) for name in page_columns]
    + list(page_dataset.schema.field("lines").type.value_type)
)
status_counts = Counter()
line_count = 0
with pq.ParquetWriter(temporary_path, flat_schema, compression="zstd") as writer:
    for batch in page_dataset.to_batches(columns=[*page_columns, "status", "lines"]):
        status_counts.update(batch.column("status").to_pylist())
        lines = batch.column("lines")
        parents = pc.list_parent_indices(lines)
        values = pc.list_flatten(lines)
        if len(values):
            arrays = [pc.take(batch.column(name), parents) for name in page_columns]
            table = pa.Table.from_arrays(
                arrays + [values.field(name) for name in line_columns],
                schema=flat_schema,
            )
            writer.write_table(table)
            line_count += table.num_rows

temporary_path.replace(flat_path)
print(f"{line_count:,} anchors written to {flat_path}")
print(f"Page status counts: {dict(status_counts)}")